In [1]:
! rm -rf /content/*

In [ ]:
"""
This script implements a machine learning pipeline for classifying handwritten digits from the MNIST dataset using
a Random Forest classifier. The pipeline includes data loading, image augmentation, model training, evaluation, and
result visualization.

The code performs the following steps:

1. **Load MNIST Dataset**: The MNIST dataset is fetched from OpenML, and the images are reshaped into a 28x28 format.

2. **Image Augmentation**: A function `augment_image` is defined to perform basic augmentations on the images,
     including horizontal and vertical flips, as well as a 90-degree clockwise rotation.

3. **Generate Augmented Dataset**: The script iterates through the original dataset, appending each original image and
     its augmentations to new lists. It also prints progress updates every 10,000 images processed.

4. **Data Preparation**: The augmented images and labels are converted to NumPy arrays and flattened for machine learning. The dataset is then split into training, validation, and test sets using stratified sampling.

5. **Print Dataset Sizes**: The sizes of the original, augmented, training, validation, and test datasets are printed
   for reference.

6. **Train Random Forest Classifier**: A Random Forest classifier is trained using GridSearchCV to find the best
     hyperparameters. The best model is saved to a file for future use.

7. **Learning Curve Visualization**: The learning curve is plotted to visualize the training and validation accuracy
     as a function of the training size.

8. **Model Evaluation**: The model is evaluated on the validation and test sets. Confusion matrices are displayed,
     and classification reports are printed to assess model performance.

9. **Download Results**: The best model is zipped and prepared for download.

10. **Shutdown Runtime**: The script includes a command to shut down the Colab runtime after the file download is
      triggered.

Dependencies:
- numpy
- opencv-python
- matplotlib
- scikit-learn
- joblib

Usage:
Run the script in a Python environment with the required libraries installed. The output will include model training
results, visualizations, and a downloadable file containing the best-trained model.
"""


import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, learning_curve, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib

# Step 1: Load MNIST
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X, y = mnist["data"], mnist["target"].astype(np.uint8)
X = X.reshape(-1, 28, 28)

# Step 2: Augmentation function (flip & rotate only)
def augment_image(img):
    augmented = []
    img_h = cv2.flip(img, 1)  # Horizontal flip
    img_v = cv2.flip(img, 0)  # Vertical flip
    img_rot = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)  # Rotate 90 deg

    augmented.extend([img_h, img_v, img_rot])
    return augmented

# Step 3: Generate augmented dataset
aug_images = []
aug_labels = []

for i in range(len(X)):
    orig = X[i]
    label = y[i]

    # Append original
    aug_images.append(orig)
    aug_labels.append(label)

    # Append augmentations
    for aug in augment_image(orig):
        aug_images.append(aug)
        aug_labels.append(label)

    if (i + 1) % 10000 == 0:
        print(f"Processed {i+1} / {len(X)}")

# Convert to array
aug_images = np.array(aug_images)
aug_labels = np.array(aug_labels)

# Flatten images for ML
aug_flat = aug_images.reshape(len(aug_images), -1)

# Step 4: Stratified split (60/20/20)
X_temp, X_test, y_temp, y_test = train_test_split(
    aug_flat, aug_labels, test_size=0.2, stratify=aug_labels, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42
)

# Step 5: Print dataset sizes
print("Original dataset size:", len(X))
print("Total after augmentation:", len(aug_flat))
print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))

# Step 6: Train RandomForest
param_grid = {
    "n_estimators": [50, 100, 150, 200, 250],
    "max_depth": [50, 100, 150, 200, 250, 300],
    # "min_samples_split": [20, 50, 100],
    # "min_samples_leaf": [1, 2, 4],
    "min_samples_leaf": [10, 20, 40],
    "max_features": ["sqrt", "log2"],
    # "bootstrap": [True, False],
    "bootstrap": [True],
    # "class_weight": [None, "balanced"]
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)
grid = GridSearchCV(rf, param_grid, cv=3, scoring="accuracy", verbose=1)
grid.fit(X_train, y_train)

# Save best model
best_model = grid.best_estimator_
print("Best parameters:", grid.best_params_)
joblib.dump(best_model, "best_fine_tune_random_forest.joblib")

# Step 7: Learning Curve
train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train, y_train, cv=3, scoring="accuracy"
)
plt.plot(train_sizes, train_scores.mean(axis=1), label="Train")
plt.plot(train_sizes, val_scores.mean(axis=1), label="Validation")
plt.title("Learning Curve")
plt.xlabel("Training Size")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.savefig("random_forest_learning_curve.png")
plt.show()

# Step 8: Evaluation
def show_confusion(title, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm)
    disp.plot()
    plt.title(title)
    plt.show()
    return cm

y_val_pred = best_model.predict(X_val)
y_test_pred = best_model.predict(X_test)

val_cm = show_confusion("Validation Confusion Matrix", y_val, y_val_pred)
test_cm = show_confusion("Test Confusion Matrix", y_test, y_test_pred)

print("Validation report:\n", classification_report(y_val, y_val_pred))
print("Test report:\n", classification_report(y_test, y_test_pred))

print("Validation Confusion Matrix:\n", val_cm)
print("Test Confusion Matrix:\n", test_cm)



# STEP 7: Zip Results
zip_filename = "best_fine_tune_random_forest.joblib"
# Trigger download
files.download(zip_filename)

# STEP 8: Shut down Colab runtime
import IPython
print("✅ Training complete. Runtime will shut down after file is downloaded.")
IPython.display.display(IPython.display.Javascript('''
  async function shutdown() {
    await google.colab.kernel.invokeFunction('shutdown');
  }
  shutdown();
'''))
